### Silver Layyer

In [0]:
from pyspark.sql.functions import *

In [0]:
# Medallion architecture ----
# Load data from bronze layer , perform transformation then write the data to silver layer

#### Read the data 

In [0]:
# Customers Table
customers_df = spark.table('workspace.bronze_layer.customers')

# Order_items Table
order_items_df = spark.table('workspace.bronze_layer.order_items')

# Orders Table
orders_df = spark.table('workspace.bronze_layer.orders')

# Products Table
products_df = spark.table('workspace.bronze_layer.products')

# Sellers table
sellers_df = spark.table('workspace.bronze_layer.sellers')

#### Transformation

In [0]:
# Chceck the records in each table
customers_df.display()
order_items_df.display()
orders_df.display()
products_df.display()
sellers_df.display()

#### Data Cleaning

In [0]:
# Remove duplicate records
customers_df = customers_df.drop_duplicates(subset=["customer_id"])
products_df = products_df.drop_duplicates(subset=["product_id"])
sellers_df = sellers_df.drop_duplicates(subset=["seller_id"])
orders_df = orders_df.drop_duplicates(subset=["order_id"])

In [0]:
# Convert order timestamp
orders_df = orders_df.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp")))

In [0]:
# Keep only delivered orders
orders_df = orders_df.filter(col("order_status") == "delivered")

In [0]:
# Calculate total order value
order_items_df = order_items_df.withColumn(
    "total_amount",
    col("price") + col("freight_value"))

#### Write the data

In [0]:
# Customers Dataset
customers_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.silver_layer.customers")

# Order_items Dataset
order_items_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.silver_layer.order_items")

# Orders Dataset
orders_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.silver_layer.orders")

# Products Dataset
products_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.silver_layer.products")

# Sellers Dataset
sellers_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.silver_layer.sellers")

